# Fase 2 — Ingesta de datos (etapas 1 y 2 del pipeline)

Proyecto: Detección temprana de pérdida de paridad de USDC en Binance (eventos Terra, FTX y SVB, 2022–2023).

Este notebook **documenta y verifica** la ingesta. No descarga ni procesa nada por defecto: usa los scripts que ya existen en `scripts/` y comprueba su resultado. Solo si se pone `EJECUTAR_DESCARGA = True` en la primera celda de código se llama a esos scripts.

| Etapa | Qué hace | Script |
|---|---|---|
| 1 | Descarga los archivos de la fuente y verifica su SHA-256 | `scripts/download_klines.py`, `scripts/download_aggtrades.py` |
| 2 | Sube los archivos a la zona raw (bronce) de HDFS | `hdfs dfs -put` (klines) y `scripts/upload_aggtrades.sh` (aggTrades) |

## 1. Qué se ingiere

**Fuente:** archivos públicos de [data.binance.vision](https://data.binance.vision) (mercado spot). No se usa la API REST.

| Dato | Pares | Granularidad | Rango |
|---|---|---|---|
| Klines (velas) | `USDCUSDT`, `BTCUSDC`, `BTCUSDT` | 1 hora, un `.zip` por mes | 2022-01 a 2023-12 (24 meses) |
| Klines sustitutos | `BUSDUSDT` (de `USDCUSDT`) y `BTCBUSD` (de `BTCUSDC`) | 1 hora | 2022-10 a 2023-02 (mensual) y 2023-03-10 (diario) |
| aggTrades | `USDCUSDT` y `BUSDUSDT` | operaciones agregadas, un `.zip` por día | solo las ventanas de evento (14 archivos) |

**Ventanas de los tres eventos** (las mismas del análisis exploratorio):

| Evento | Ventana EDA | Archivos aggTrades |
|---|---|---|
| Terra | 2022-05-07 a 2022-05-17 | `USDCUSDT` del 9 al 13 de mayo de 2022 |
| FTX | 2022-11-05 a 2022-11-15 | `BUSDUSDT` del 7 al 11 de noviembre de 2022 |
| SVB | 2023-03-08 a 2023-03-18 | `BUSDUSDT` el 10 de marzo de 2023 y `USDCUSDT` del 11 al 13 de marzo de 2023 |

**Por qué existen los sustitutos.** Según el Anexo B.1 de la guía operativa y el encabezado de `scripts/download_klines.py`, `USDCUSDT` y `BTCUSDC` no tienen datos en la fuente entre el 27 de septiembre de 2022 y el 10–11 de marzo de 2023: Binance convirtió USDC, USDP y TUSD a BUSD en septiembre de 2022. En ese hueco se usa `BUSDUSDT` como sustituto de `USDCUSDT`, y `BTCBUSD` como sustituto de `BTCUSDC`. Los datos sustitutos quedan marcados en la columna `fuente_paridad` y **no se mezclan sin marcar** con los reales; por la misma razón, el 10 de marzo de 2023 (día de la quiebra de SVB) la serie viene del sustituto hasta que `USDCUSDT` vuelve a tener operaciones el 11 de marzo.

## 2. Parámetros y descarga opcional

`EJECUTAR_DESCARGA` vale `False`: **por defecto no se descarga ni se sube nada**. Con `True`, la celda llama con `subprocess` a los tres scripts de la ingesta, en el orden del pipeline (necesita red y HDFS en marcha).

Rutas que se buscan (se pueden fijar con las variables de entorno `ZONA_CRUDA_DIR` y `AGGTRADES_LOCAL`): `zona_cruda/` en la raíz del repositorio (klines y `missing.log`) y `~/aggtrades_local` (zips y CSV de aggTrades, que es donde los deja `download_aggtrades.py`).

In [1]:
import os, sys, subprocess, shutil, hashlib, zipfile, urllib.request
from pathlib import Path
import pandas as pd

# Opciones de visualizacion de pandas.
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 200)

# Interruptores del notebook (ver la celda de texto de arriba).
EJECUTAR_DESCARGA = False          # True solo para repetir la ingesta (usa red y HDFS)
CONSULTAR_CHECKSUM_PUBLICADO = True  # compara con el .CHECKSUM de la fuente (archivos de pocos bytes); sin red se omite

# Raiz del repositorio: funciona si el notebook se abre desde notebooks/ o desde la raiz.
CWD = Path.cwd()
REPO = CWD.parent if CWD.name == "notebooks" else CWD

# Devuelve la primera carpeta de la lista que exista, o None.
def primera_existente(candidatas):
    for c in candidatas:
        if c and Path(c).is_dir():
            return Path(c)
    return None

# Copias locales de la descarga (variable de entorno o carpeta por defecto) y evidencia versionada.
ZONA_CRUDA = primera_existente([os.environ.get("ZONA_CRUDA_DIR"), REPO / "zona_cruda", Path.home() / "zona_cruda"])
AGG_LOCAL = primera_existente([os.environ.get("AGGTRADES_LOCAL"), Path.home() / "aggtrades_local"])
EVIDENCIA_AGG = REPO / "benchmarks" / "resultados" / "aggtrades" / "descarga_resumen.tsv"

print("Repositorio       :", REPO)
print("zona_cruda        :", ZONA_CRUDA or "no se encontró copia local")
print("aggtrades_local   :", AGG_LOCAL or "no se encontró copia local")
print("Evidencia en repo :", EVIDENCIA_AGG, "(existe)" if EVIDENCIA_AGG.exists() else "(no existe)")
print("EJECUTAR_DESCARGA =", EJECUTAR_DESCARGA)

Repositorio       : /home/marlon/pedido12/stablecoin-depeg-detection
zona_cruda        : /home/marlon/pedido12/stablecoin-depeg-detection/zona_cruda
aggtrades_local   : /home/marlon/aggtrades_local
Evidencia en repo : /home/marlon/pedido12/stablecoin-depeg-detection/benchmarks/resultados/aggtrades/descarga_resumen.tsv (existe)
EJECUTAR_DESCARGA = False


In [2]:
# Repite la ingesta (descarga de klines, descarga de aggTrades y subida a HDFS) solo si
# EJECUTAR_DESCARGA = True. Se detiene en el primer paso que termine con error.
if EJECUTAR_DESCARGA:
    pasos = [
        [sys.executable, "scripts/download_klines.py"],
        [sys.executable, "scripts/download_aggtrades.py"],
        ["bash", "scripts/upload_aggtrades.sh"],
    ]
    for cmd in pasos:
        print("$", " ".join(cmd), flush=True)
        r = subprocess.run(cmd, cwd=REPO)
        print("código de salida:", r.returncode)
        if r.returncode != 0:
            print("Se detiene: el paso anterior terminó con error.")
            break
else:
    print("Descarga omitida (EJECUTAR_DESCARGA = False). No se descargó ni se subió nada.")

Descarga omitida (EJECUTAR_DESCARGA = False). No se descargó ni se subió nada.


## 3. Inventario local

Cuenta los `.zip` que hay en disco por símbolo y tipo de dato, y muestra `missing.log`, que lista los archivos que **no existen en la fuente** (o cuyo checksum no coincidió). Si no hay copia local (por ejemplo en un clon limpio), el inventario de aggTrades se toma de la evidencia que ya está versionada en el repositorio (`benchmarks/resultados/aggtrades/descarga_resumen.tsv`).

In [3]:
# Inventario: cuantos .zip y cuantos MB hay por tipo de dato y simbolo.
filas = []
if ZONA_CRUDA and (ZONA_CRUDA / "klines").is_dir():
    for d in sorted((ZONA_CRUDA / "klines").iterdir()):
        if d.is_dir():
            zs = sorted(d.glob("*.zip"))
            filas.append(dict(tipo="klines 1h", simbolo=d.name, zips=len(zs), MB=round(sum(z.stat().st_size for z in zs) / 1e6, 1), origen="local"))
if AGG_LOCAL and (AGG_LOCAL / "zip").is_dir():
    for d in sorted((AGG_LOCAL / "zip").iterdir()):
        if d.is_dir():
            zs = sorted(d.glob("*.zip"))
            filas.append(dict(tipo="aggTrades diarios", simbolo=d.name, zips=len(zs), MB=round(sum(z.stat().st_size for z in zs) / 1e6, 1), origen="local"))
# Sin copia local de aggTrades: se usa el resumen de descarga versionado en el repositorio.
elif EVIDENCIA_AGG.exists():
    ev = pd.read_csv(EVIDENCIA_AGG, sep="\t")
    for sim, g in ev.groupby("simbolo"):
        filas.append(dict(tipo="aggTrades diarios", simbolo=sim, zips=len(g), MB=round(g["zip_bytes"].sum() / 1e6, 1), origen="evidencia del repo"))

inv = pd.DataFrame(filas)
print(inv.to_string(index=False) if len(inv) else "Sin inventario: no hay copia local ni evidencia.")
print()

# Archivos que no existen en la fuente (o con checksum invalido).
for nombre, ruta in [("zona_cruda/missing.log", (ZONA_CRUDA / "missing.log") if ZONA_CRUDA else None),
                     ("data/missing.log (versionado en el repo)", REPO / "data" / "missing.log")]:
    if ruta and ruta.exists():
        lineas = [l for l in ruta.read_text().splitlines() if l.strip()]
        print(f"{nombre}: {len(lineas)} líneas")
        for l in lineas:
            print("   ", l)
    else:
        print(f"{nombre}: no existe")

             tipo  simbolo  zips   MB origen
        klines 1h  BTCBUSD     5  0.2  local
        klines 1h  BTCUSDC    19  0.7  local
        klines 1h  BTCUSDT    24  1.0  local
        klines 1h BUSDUSDT     5  0.1  local
        klines 1h USDCUSDT    19  0.5  local
aggTrades diarios BUSDUSDT     6 72.4  local
aggTrades diarios USDCUSDT     8 38.1  local

zona_cruda/missing.log: 10 líneas
    USDCUSDT-1h-2022-10.zip
    USDCUSDT-1h-2022-11.zip
    USDCUSDT-1h-2022-12.zip
    USDCUSDT-1h-2023-01.zip
    USDCUSDT-1h-2023-02.zip
    BTCUSDC-1h-2022-10.zip
    BTCUSDC-1h-2022-11.zip
    BTCUSDC-1h-2022-12.zip
    BTCUSDC-1h-2023-01.zip
    BTCUSDC-1h-2023-02.zip
data/missing.log (versionado en el repo): 0 líneas


## 4. Verificación de integridad (SHA-256)

Para cada `.zip` local se hace tres cosas:

1. Se recalcula su **SHA-256**.
2. Se compara con el `.CHECKSUM` publicado por la fuente. Los scripts de descarga leen ese `.CHECKSUM` y lo descartan después de verificar, así que no queda guardado junto al `.zip`; por eso el notebook vuelve a consultarlo (un archivo de unos 100 bytes por cada `.zip`). Si ya hubiera un `<archivo>.zip.CHECKSUM` local se usa ese. Sin red, el archivo queda como «sin comparar».
3. Se prueba que el `.zip` se abra y que su CRC interno sea correcto (`zipfile.testzip`).

In [4]:
BASE = "https://data.binance.vision/data/spot"

# SHA-256 de un archivo, leido en bloques de 1 MiB.
def sha256_de(ruta):
    h = hashlib.sha256()
    with open(ruta, "rb") as f:
        for b in iter(lambda: f.read(1 << 20), b""):
            h.update(b)
    return h.hexdigest()

# URL del .CHECKSUM publicado para un .zip (diario o mensual, segun el nombre del archivo).
def url_checksum(zip_path, tipo):
    nombre, sim = zip_path.name, zip_path.parent.name
    if tipo == "aggTrades":
        return f"{BASE}/daily/aggTrades/{sim}/{nombre}.CHECKSUM"
    diario = len(nombre.replace(".zip", "").split("-")) == 5   # SIMBOLO-1h-AAAA-MM-DD
    return f"{BASE}/{'daily' if diario else 'monthly'}/klines/{sim}/1h/{nombre}.CHECKSUM"

# Checksum esperado: primero se busca junto al .zip; si no, se consulta la fuente (si hay red).
def checksum_publicado(zip_path, tipo):
    local = Path(str(zip_path) + ".CHECKSUM")
    if local.exists():
        return local.read_text().split()[0].lower(), "archivo local"
    if not CONSULTAR_CHECKSUM_PUBLICADO:
        return None, "no consultado"
    try:
        req = urllib.request.Request(url_checksum(zip_path, tipo), headers={"User-Agent": "Mozilla/5.0"})
        with urllib.request.urlopen(req, timeout=15) as r:
            return r.read().decode().split()[0].lower(), "fuente"
    except Exception as e:
        return None, f"sin red ({type(e).__name__})"

# Lista de .zip locales de klines y de aggTrades.
zips = []
if ZONA_CRUDA and (ZONA_CRUDA / "klines").is_dir():
    zips += [(p, "klines") for p in sorted((ZONA_CRUDA / "klines").glob("*/*.zip"))]
if AGG_LOCAL and (AGG_LOCAL / "zip").is_dir():
    zips += [(p, "aggTrades") for p in sorted((AGG_LOCAL / "zip").glob("*/*.zip"))]

# Para cada .zip: SHA-256 real, checksum esperado y prueba de integridad interna del zip (CRC).
res = []
sin_red = 0
for p, tipo in zips:
    real = sha256_de(p)
    esperado, via = checksum_publicado(p, tipo)
    if esperado is None and via.startswith("sin red"):
        sin_red += 1
        if sin_red >= 3:                         # sin red: no se reintenta archivo por archivo
            CONSULTAR_CHECKSUM_PUBLICADO = False
    try:
        crc_ok = zipfile.ZipFile(p).testzip() is None
    except Exception:
        crc_ok = False
    estado = "sin comparar" if esperado is None else ("coincide" if esperado == real else "NO coincide")
    res.append(dict(tipo=tipo, simbolo=p.parent.name, archivo=p.name, sha256=real[:16] + "…", checksum=estado, via=via, zip_ok=crc_ok))

# Resumen global y por simbolo.
ver = pd.DataFrame(res)
if len(ver):
    resumen_ver = pd.DataFrame({
        "total de .zip": [len(ver)],
        "SHA-256 coincide": [int((ver.checksum == "coincide").sum())],
        "NO coincide": [int((ver.checksum == "NO coincide").sum())],
        "sin comparar": [int((ver.checksum == "sin comparar").sum())],
        "zip con CRC correcto": [int(ver.zip_ok.sum())],
    })
    print(resumen_ver.to_string(index=False))
    print()
    print(ver.groupby(["tipo", "simbolo"]).agg(archivos=("archivo", "count"), coinciden=("checksum", lambda s: int((s == "coincide").sum())),
                                              no_coinciden=("checksum", lambda s: int((s == "NO coincide").sum())),
                                              sin_comparar=("checksum", lambda s: int((s == "sin comparar").sum()))).reset_index().to_string(index=False))
    if (ver.checksum == "NO coincide").any():
        print("\nArchivos que NO coinciden:")
        print(ver[ver.checksum == "NO coincide"].to_string(index=False))
else:
    print("No hay .zip locales en esta máquina (por ejemplo, un clon limpio): no se puede recalcular el SHA-256.")
    print("La verificación hecha en la descarga queda registrada en benchmarks/resultados/aggtrades/descarga_resumen.tsv (columna checksum).")

 total de .zip  SHA-256 coincide  NO coincide  sin comparar  zip con CRC correcto
            86                86            0             0                    86

     tipo  simbolo  archivos  coinciden  no_coinciden  sin_comparar
aggTrades BUSDUSDT         6          6             0             0
aggTrades USDCUSDT         8          8             0             0
   klines  BTCBUSD         5          5             0             0
   klines  BTCUSDC        19         19             0             0
   klines  BTCUSDT        24         24             0             0
   klines BUSDUSDT         5          5             0             0
   klines USDCUSDT        19         19             0             0


## 5. Estado de la capa bronce en HDFS

Muestra lo que hay en `/datalake/raw/binance/klines` y `/datalake/raw/binance/aggtrades` con `hdfs dfs -ls -R` y `hdfs dfs -du -s -h`. Solo lee; no escribe nada. Si HDFS no está en marcha, la celda lo dice y el notebook sigue.

In [5]:
# Ejecuta 'hdfs dfs ...' y devuelve la salida como texto. Si HDFS no esta disponible, lo informa sin fallar.
def hdfs(*args):
    if shutil.which("hdfs") is None:
        return "(no se encontró el comando hdfs en esta máquina)"
    try:
        r = subprocess.run(["hdfs", "dfs", *args], capture_output=True, text=True, timeout=180)
        return (r.stdout + r.stderr).strip() or "(sin salida)"
    except Exception as e:
        return f"(no se pudo consultar HDFS: {type(e).__name__}: {e})"

# Solo lectura: listado recursivo y tamano total de la capa bronce (zona raw).
for ruta in ["/datalake/raw/binance/klines", "/datalake/raw/binance/aggtrades"]:
    print("=" * 100)
    print("$ hdfs dfs -ls -R", ruta)
    print(hdfs("-ls", "-R", ruta))
    print()
    print("$ hdfs dfs -du -s -h", ruta)
    print(hdfs("-du", "-s", "-h", ruta))

$ hdfs dfs -ls -R /datalake/raw/binance/klines


drwxr-xr-x   - marlon supergroup          0 2026-11-16 14:15 /datalake/raw/binance/klines/BTCBUSD
-rw-r--r--   4 marlon supergroup     119456 2026-09-23 16:49 /datalake/raw/binance/klines/BTCBUSD/BTCBUSD-1h-2022-10.csv
-rw-r--r--   1 marlon supergroup      44129 2026-11-16 14:15 /datalake/raw/binance/klines/BTCBUSD/BTCBUSD-1h-2022-10.zip
-rw-r--r--   4 marlon supergroup     115965 2026-09-23 16:49 /datalake/raw/binance/klines/BTCBUSD/BTCBUSD-1h-2022-11.csv
-rw-r--r--   1 marlon supergroup      43016 2026-11-16 14:15 /datalake/raw/binance/klines/BTCBUSD/BTCBUSD-1h-2022-11.zip
-rw-r--r--   4 marlon supergroup     119520 2026-09-23 16:49 /datalake/raw/binance/klines/BTCBUSD/BTCBUSD-1h-2022-12.csv
-rw-r--r--   1 marlon supergroup      43813 2026-11-16 14:15 /datalake/raw/binance/klines/BTCBUSD/BTCBUSD-1h-2022-12.zip
-rw-r--r--   4 marlon supergroup     120047 2026-09-23 16:49 /datalake/raw/binance/klines/BTCBUSD/BTCBUSD-1h-2023-01.csv
-rw-r--r--   1 marlon supergroup      44426 2026-11-16 

10.2 M  33.1 M  /datalake/raw/binance/klines
$ hdfs dfs -ls -R /datalake/raw/binance/aggtrades


drwxr-xr-x   - marlon supergroup          0 2026-11-16 14:15 /datalake/raw/binance/aggtrades/BUSDUSDT
-rw-r--r--   2 marlon supergroup    6758936 2026-11-16 14:15 /datalake/raw/binance/aggtrades/BUSDUSDT/BUSDUSDT-aggTrades-2022-11-07.zip
-rw-r--r--   2 marlon supergroup   13953334 2026-11-16 14:15 /datalake/raw/binance/aggtrades/BUSDUSDT/BUSDUSDT-aggTrades-2022-11-08.zip
-rw-r--r--   2 marlon supergroup   14464361 2026-11-16 14:15 /datalake/raw/binance/aggtrades/BUSDUSDT/BUSDUSDT-aggTrades-2022-11-09.zip
-rw-r--r--   2 marlon supergroup   18846050 2026-11-16 14:15 /datalake/raw/binance/aggtrades/BUSDUSDT/BUSDUSDT-aggTrades-2022-11-10.zip
-rw-r--r--   2 marlon supergroup   11424945 2026-11-16 14:15 /datalake/raw/binance/aggtrades/BUSDUSDT/BUSDUSDT-aggTrades-2022-11-11.zip
-rw-r--r--   2 marlon supergroup    7000630 2026-11-16 14:15 /datalake/raw/binance/aggtrades/BUSDUSDT/BUSDUSDT-aggTrades-2023-03-10.zip
drwxr-xr-x   - marlon supergroup          0 2026-11-16 14:15 /datalake/raw/binance

693.3 M  1.4 G  /datalake/raw/binance/aggtrades


## 6. Resumen

Una línea con el resultado de la descarga de aggTrades: archivos esperados, descargados, con problema y registros totales. Sale de `descarga_resumen.tsv` (el de `~/aggtrades_local` si existe; si no, el versionado en el repositorio). Los valores esperados son 14, 14, 0 y 7 965 158.

In [6]:
# Usa el descarga_resumen.tsv local si existe; si no, el versionado en el repositorio.
fuente_resumen = None
if AGG_LOCAL and (AGG_LOCAL / "descarga_resumen.tsv").exists():
    fuente_resumen = AGG_LOCAL / "descarga_resumen.tsv"
elif EVIDENCIA_AGG.exists():
    fuente_resumen = EVIDENCIA_AGG

if fuente_resumen is None:
    print("No hay descarga_resumen.tsv ni local ni en el repositorio.")
else:
    t = pd.read_csv(fuente_resumen, sep="\t")
    ok = t[t["checksum"].isin(["OK", "SIN_CHECKSUM"])]
    esperados, descargados, problema, registros = len(t), len(ok), len(t) - len(ok), int(ok["registros"].sum())
    print(f"archivos esperados: {esperados} | descargados: {descargados} | con problema: {problema} | registros totales de aggTrades: {registros:,}")
    print(f"(fuente: {fuente_resumen})")
    # Comparacion con los valores esperados del informe.
    coincide = (esperados, descargados, problema, registros) == (14, 14, 0, 7965158)
    print("Valores esperados: 14 | 14 | 0 | 7,965,158 ->", "COINCIDEN" if coincide else "DIFERENTES (reportar tal cual)")

archivos esperados: 14 | descargados: 14 | con problema: 0 | registros totales de aggTrades: 7,965,158
(fuente: /home/marlon/aggtrades_local/descarga_resumen.tsv)
Valores esperados: 14 | 14 | 0 | 7,965,158 -> COINCIDEN
